# 🧠 DeepStack Value Network Training

**Championship-Grade Neural Network Training**

This notebook trains the DeepStack value network that estimates counterfactual
values for terminal nodes in depth-limited lookahead.

Features:
- Load pre-generated CFR-solved training samples
- Train 4-layer MLP (12K+ parameters per DeepStack paper)
- Temperature scaling for calibration
- ONNX export for optimized inference

## Requirements
- Pre-generated CFR samples (cfr_samples.npz)
- Runtime Type: GPU (optional but faster)

## 1️⃣ Setup & Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

CONFIG = {
    # Data paths
    "cfr_samples": "/content/drive/MyDrive/poker_ai/data/cfr_samples.npz",
    "output_dir": "/content/drive/MyDrive/poker_ai/models",
    "model_name": "deepstack_champion.pt",
    
    # Architecture (matches DeepStack paper)
    "input_size": 36,
    "hidden_layers": [256, 128, 64],
    "output_size": 36,
    "dropout": 0.1,
    
    # Training settings
    "batch_size": 256,
    "learning_rate": 1e-3,
    "num_epochs": 100,
    "weight_decay": 1e-5,
    "early_stopping_patience": 10,
    "val_split": 0.1,
    
    # Calibration
    "apply_temperature_scaling": True,
    "target_ece": 0.05,
    
    # ONNX export
    "export_onnx": True,
}

print("✅ Configuration loaded")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2️⃣ Load Training Data

In [ ]:
# Load CFR-solved samples
if os.path.exists(CONFIG["cfr_samples"]):
    data = np.load(CONFIG["cfr_samples"])
    
    # Expected format:
    # - public_features: [N, input_size] - pot, street, board encoding
    # - opponent_range: [N, num_hands] - opponent probability distribution
    # - cfv_targets: [N, num_hands] - counterfactual values
    
    public_features = data.get('public_features', data.get('features', None))
    opponent_ranges = data.get('opponent_range', data.get('ranges', None))
    cfv_targets = data.get('cfv_targets', data.get('values', None))
    
    if public_features is not None and cfv_targets is not None:
        print(f"Loaded {len(public_features)} training samples")
        print(f"Features shape: {public_features.shape}")
        print(f"Targets shape: {cfv_targets.shape}")
    else:
        print("⚠️ Data format not recognized. Generating synthetic data...")
        public_features = None
else:
    print("⚠️ CFR samples not found. Generating synthetic data...")
    public_features = None

In [ ]:
# Generate synthetic training data if needed
if public_features is None:
    print("Generating synthetic CFR data for demonstration...")
    
    N = 10000  # Number of samples
    input_size = CONFIG["input_size"]
    
    # Generate random game states
    np.random.seed(42)
    public_features = np.random.randn(N, input_size).astype(np.float32)
    opponent_ranges = np.random.dirichlet(np.ones(input_size), N).astype(np.float32)
    
    # Generate plausible CFV targets
    # Real data would come from CFR solving
    cfv_targets = np.zeros((N, input_size), dtype=np.float32)
    for i in range(N):
        # Simulate equity-based values
        equity = np.random.beta(2, 2, input_size)
        pot = np.abs(public_features[i, 0]) * 100 + 50
        cfv_targets[i] = pot * (2 * equity - 1)
    
    print(f"Generated {N} synthetic samples")

In [ ]:
# Prepare input features (concatenate public state + opponent range)
if opponent_ranges is not None:
    X = np.concatenate([public_features, opponent_ranges], axis=1)
else:
    # If no ranges, just use public features
    X = public_features
    # Pad to expected input size
    if X.shape[1] < CONFIG["input_size"] * 2:
        padding = np.zeros((X.shape[0], CONFIG["input_size"] * 2 - X.shape[1]))
        X = np.concatenate([X, padding], axis=1)

y = cfv_targets

print(f"Input shape: {X.shape}")
print(f"Output shape: {y.shape}")

In [ ]:
# Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# Create dataset
dataset = TensorDataset(X_tensor, y_tensor)

# Split into train/val
val_size = int(len(dataset) * CONFIG["val_split"])
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False
)

print(f"Training samples: {train_size}")
print(f"Validation samples: {val_size}")

## 3️⃣ Define Value Network

In [ ]:
class DeepStackValueNetwork(nn.Module):
    """
    DeepStack-style value network.
    
    Maps (public state, opponent range) → counterfactual values for all hands.
    """
    
    def __init__(self, input_size, hidden_layers, output_size, dropout=0.1):
        super().__init__()
        
        layers = []
        in_features = input_size * 2  # public + range
        
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_features = hidden_size
        
        layers.append(nn.Linear(in_features, output_size))
        
        self.network = nn.Sequential(*layers)
        self.temperature = nn.Parameter(torch.ones(1))
    
    def forward(self, x):
        return self.network(x)
    
    def calibrated_forward(self, x):
        return self.forward(x) / self.temperature


# Create model
model = DeepStackValueNetwork(
    input_size=CONFIG["input_size"],
    hidden_layers=CONFIG["hidden_layers"],
    output_size=CONFIG["output_size"],
    dropout=CONFIG["dropout"]
).to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")
print(f"Target: >12,000 (DeepStack paper)")
print(model)

## 4️⃣ Training Loop

In [ ]:
# Training setup
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Loss function (Huber loss for robustness)
criterion = nn.HuberLoss()

# Training tracking
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0

In [ ]:
# Training loop
print("Starting training...")
print("="*60)

for epoch in range(CONFIG["num_epochs"]):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Print progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{CONFIG['num_epochs']} | "
              f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        
        # Save best model
        best_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stopping_patience"]:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Restore best model
model.load_state_dict(best_state)
print(f"\n✅ Training complete! Best val loss: {best_val_loss:.6f}")

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('Training Progress (Log Scale)')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5️⃣ Temperature Scaling (Calibration)

In [ ]:
if CONFIG["apply_temperature_scaling"]:
    print("Applying temperature scaling for calibration...")
    
    # Get validation predictions
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            preds = model(batch_x)
            all_preds.append(preds.cpu())
            all_targets.append(batch_y)
    
    val_preds = torch.cat(all_preds, dim=0)
    val_targets = torch.cat(all_targets, dim=0)
    
    # Grid search for optimal temperature
    best_temp = 1.0
    best_error = float('inf')
    
    for temp in np.linspace(0.5, 2.0, 31):
        scaled = val_preds / temp
        error = F.huber_loss(scaled, val_targets).item()
        if error < best_error:
            best_error = error
            best_temp = temp
    
    # Set optimal temperature
    model.temperature.data = torch.tensor([best_temp])
    
    print(f"Optimal temperature: {best_temp:.3f}")
    print(f"Calibrated loss: {best_error:.6f}")

## 6️⃣ Save Model

In [ ]:
# Save PyTorch model
os.makedirs(CONFIG["output_dir"], exist_ok=True)
model_path = os.path.join(CONFIG["output_dir"], CONFIG["model_name"])

torch.save({
    'model_state_dict': model.state_dict(),
    'input_size': CONFIG["input_size"],
    'hidden_layers': CONFIG["hidden_layers"],
    'output_size': CONFIG["output_size"],
    'temperature': model.temperature.item(),
    'best_val_loss': best_val_loss,
}, model_path)

print(f"✅ Model saved to {model_path}")

In [ ]:
# Export to ONNX
if CONFIG["export_onnx"]:
    onnx_path = os.path.join(CONFIG["output_dir"], "deepstack_champion.onnx")
    
    dummy_input = torch.randn(1, CONFIG["input_size"] * 2, device=device)
    
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        opset_version=14,
        input_names=['input'],
        output_names=['values'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'values': {0: 'batch_size'}
        }
    )
    
    print(f"✅ ONNX model saved to {onnx_path}")

## 7️⃣ Validation Summary

In [ ]:
# Final validation metrics
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model Parameters: {num_params:,}")
print(f"Best Validation Loss: {best_val_loss:.6f}")
print(f"Temperature: {model.temperature.item():.3f}")
print(f"Training Epochs: {len(train_losses)}")
print(f"")
print(f"Model saved to: {model_path}")
if CONFIG["export_onnx"]:
    print(f"ONNX export: {onnx_path}")
print("="*60)